# 🚀 Build a shopping agent with Sequence-Level Knowledge Distillation on Amazon SageMaker AI

This notebook teaches a small, inexpensive model to act as an agentic **shopping assistant** by learning from the decoded trajectories of a stronger "teacher" agent with a technique called **Sequence-Level Knowledge Distillation (SeqKD)**.

### Why distillation?
A large frontier model can solve agentic tool-use tasks well, but is slow and expensive to run in production. SeqKD lets you transfer that capability into a small model that is cheaper to host and fast to serve, while retaining most of the teacher's skill on the task. SeqKD does this by running **supervised fine-tuning (SFT)** on the teacher generated trajectories. We run the fine-tuning with SageMaker's **serverless `SFTTrainer`** (SageMaker Python SDK v3), which handles the training infrastructure for you.

### The task
A shopper asks for products in natural language — e.g. *"women's jeans in size eu:30 with a high waist, priced above 114 Pesos"*. The agent must use four tools to search a **2.7-million-product catalog**, inspect possible candidates, verify every constraint, and recommend matching products. The available tools to the agent are:

| Tool | Purpose |
|---|---|
| `find_product` | Search the catalog |
| `view_product_information` | Inspect a candidate's details |
| `recommend_product` | Return the final recommendation |
| `terminate` | End the task |

The catalog, test sets, and the success metric come from **[ShoppingBench](https://github.com/yjwjy/ShoppingBench)**.

### The teacher
A teacher model on **Amazon Bedrock** (we use GLM-5.3) plays ShoppingBench on the training problems; only the **solved** trajectories are kept, with the teacher's reasoning, and the student (a thinking model) is trained on them. You need Bedrock access to the teacher model. The full ShoppingBench setup (catalog + search index, ~15 GB) is done as part of this notebook. We use the teacher GLM-5.3 published trajectories that are referenced in this notebook by default so it saves about 1-2 hours time of teacher rollouts. To generate the rollouts from bedrock you can set `USE_PUBLISHED_TEACHER_TRAJECTORIES = False`.

The ShoppingBench test problems are split once into **training** problems (425, the teacher's prompts) and **held-out** problems (324, used only for evaluation). The split is shipped as fixed line indices in `split/` and loaded by `scripts/split.py`.

### What you will do
1. **Set up** your environment and SageMaker session.
2. **Prepare the data** — generate the teacher's trajectories, encode each step into the exact prompt/response format the ShoppingBench agent loop uses, then register it as a versioned SageMaker dataset.
3. **Train** the student with the serverless `SFTTrainer with LoRA fine-tuning (step 02) and the encoded trajectories (step 01) as input.
4. **Retrieve** the registered, deploy-ready Model Package.
5. **Evaluate** the student in the two companion notebooks: `2-tool-call-accuracy-gate.ipynb` (a cheap, fast per-step tool-call accuracy gate — run it first) and `3-evaluate-shopping-agent.ipynb` (the full ShoppingBench ASR and IFEval run).

> **Reference result (held-out, Bedrock teacher, thinking on):** the student's mean Absolute Success Rate rises from **0.297 (base) → 0.541 (student)** across the three ShoppingBench intents, slightly above its GLM-5.3 teacher (0.51) and well above the 8x larger Qwen3-32B (0.228). These numbers are from one sample run and will vary with training data, temperature, and models used.

## 00. Setup

Install the pinned dependencies and configure the SageMaker session, execution role, and S3 location used throughout the notebook. Run the install cell **first**, then restart the kernel so the pinned package versions are the ones imported.

In [ ]:
# Install dependencies, pinned to the versions resolved at authoring time (reproducible).
# sagemaker: v3 SDK (latest at authoring). huggingface_hub/transformers: the student tokenizer (step 01).
%pip install -q \
    "sagemaker==3.23.0" \
    "huggingface_hub==1.33.0" \
    "transformers==5.19.0"
# NOTE: restart the kernel after this install so the pinned versions are imported.

**Configure the session.** This cell resolves everything downstream steps depend on:

- **Region** — taken from your environment (defaults to `us-west-2`).
- **Execution role** — the IAM role SageMaker assumes to run training and register artifacts. If auto-detection fails (e.g. running locally), replace the placeholder ARN with your own.
- **S3 location** — training data and model outputs are staged under `s3://<default-bucket>/seqkd-shopping/`. We use the SageMaker session's default bucket; replace it if you prefer a specific bucket.
- **Teacher data** — `BEDROCK_TEACHER`, `TEACHER_FIELDS`, `TEACHER_SAMPLES` and `INCLUDE_REASONING` configure step 01; `USE_PUBLISHED_TEACHER_TRAJECTORIES = True` starts from the published GLM-5.3 trajectories (`teacher_trajectories/glm-5.3/`) instead of calling Bedrock.

In [ ]:
import os, json, boto3
from sagemaker.core.helper.session_helper import Session, get_execution_role

REGION = boto3.Session().region_name or "us-west-2"
os.environ["AWS_DEFAULT_REGION"] = os.environ["AWS_REGION"] = REGION

# SageMaker session (SDK v3)
sagemaker_session = Session(boto_session=boto3.Session(region_name=REGION))

try:
    ROLE = get_execution_role()
    print("Execution role:", ROLE)
except Exception:
    ACCOUNT = boto3.client("sts").get_caller_identity()["Account"]
    ROLE = f"arn:aws:iam::{ACCOUNT}:role/<your-sagemaker-execution-role>"   #replace with your unique execution role

# Use the SageMaker session's default bucket; fall back to a placeholder if unavailable
try:
    BUCKET = sagemaker_session.default_bucket()
except Exception:
    BUCKET = "<your-bucket>" #replace with your bucket
    
PREFIX  = "seqkd-shopping"
S3_ROOT = f"s3://{BUCKET}/{PREFIX}"

# ---- Teacher data (step 01) ----
BEDROCK_TEACHER   = "us.zai.glm-5.3"                 # Bedrock model or inference-profile ID of the teacher
TEACHER_FIELDS    = {"reasoning_effort": "high"}     # additionalModelRequestFields: turns the teacher's reasoning on
TEACHER_SAMPLES   = 2                                # trajectories generated per training query
INCLUDE_REASONING = True                             # keep the teacher's reasoning in the targets (a thinking student)

# True: start from the published GLM-5.3 trajectories in teacher_trajectories/glm-5.3/ (the run behind the reported
# results; 2 samples per training problem) instead of calling Bedrock. Generation then skips every problem they cover.
USE_PUBLISHED_TEACHER_TRAJECTORIES = True
print(REGION, ROLE, S3_ROOT)

## 01. Data preparation

The student learns by imitating the teacher's successful tool-use trajectories. In this step we (a) fetch the ShoppingBench harness and load the training problems, (b) have the teacher play those problems and encode each step of its solved trajectories into a `{prompt, completion}` training example, and (c) register the result as versioned SageMaker datasets.

### 01a. Fetch the ShoppingBench harness and load the training problems

The cell below runs `scripts/setup_shopbench.sh --code-only`, which checks out the **ShoppingBench** repo at a **pinned commit** — so the harness code we execute is a fixed, reviewable snapshot for reproducibility. It then loads the **training problems** from the fixed split (`scripts/split.py`, indices in `split/`): 147 `product`, 145 `shop` and 133 `voucher` test problems. The held-out problems that notebook 3 evaluates on are never used here. The loader checks a hash of each ShoppingBench test file, so a different ShoppingBench version fails here instead of silently changing the split.

In [ ]:
import json, sys, subprocess
from pathlib import Path

from transformers import AutoTokenizer

HF_STUDENT = "Qwen/Qwen3.5-4B"
tok = AutoTokenizer.from_pretrained(HF_STUDENT, trust_remote_code=True)   # student tokenizer (length filter, step 01)

# ---- Fetch the ShoppingBench harness (pinned commit, code only — no catalog/index) ----
# Paths are relative to this notebook's folder (the kernel runs here).
SHOPBENCH_WORK = Path("shopbench-eval")
SHOPBENCH_WORK.mkdir(parents=True, exist_ok=True)
subprocess.run(["bash", "scripts/setup_shopbench.sh", "--code-only", str(SHOPBENCH_WORK)], check=True)
SB_DIR = SHOPBENCH_WORK / "ShoppingBench"

# ---- Load the training problems from the fixed split (the held-out problems are for notebook 3 only) ----
sys.path.insert(0, str(Path("scripts").resolve()))
from split import load_split, write_jsonl

train_problems = {t: load_split(SB_DIR / "data", t)[0] for t in ("product", "shop", "voucher")}
for t, rows in train_problems.items():
    print(f"{t}: {len(rows)} training problems")

### 01b. Generate the trajectories with a Bedrock teacher

A teacher model on **Amazon Bedrock** plays ShoppingBench's agent on the training problems, and only the trajectories the benchmark scores as **solved** become training data (rejection sampling). The teacher's **reasoning** is kept as a `<think>…</think>` block in each target, so the student learns *why* the teacher calls a particular tool, not just which one.

- **Training problems:** the 425 problems loaded in 01a.
- **Teacher:** `BEDROCK_TEACHER` through the local relay's Bedrock transport (Converse API). `TEACHER_FIELDS` turns reasoning on; the relay returns it as `<think>…</think>`.
- **What runs locally:** the agent loop and the product search server (the catalog index is built on the first run, about an hour), so this step needs the complete ShoppingBench setup.
- **Re-running is safe:** finished trajectories are skipped, and a job that stops making progress is stopped so a malformed tool call can't hang the cell.
- **Training examples:** each step of a solved trajectory becomes one `{prompt, completion}` example: the prompt is exactly what the harness sent the teacher (the agent instructions + tool list, then the dialogue history so far), and the completion is the teacher's reasoning and next action. `SFTTrainer` computes the loss on the completion only.
- **Split by query, not by step:** each query yields several steps (and up to `TEACHER_SAMPLES` trajectories), so a step-level split would leak near-duplicates across train/val. We hold out ~5% of *queries* instead.

With `INCLUDE_REASONING = False`, the same solved trajectories are used without the `<think>` blocks.

In [ ]:
import os, random
import sm_openai_relay as relay
from encode_rollouts import sft_rows
from shopbench_harness import Harness

# Full ShoppingBench: catalog + Lucene index for the search tool (skips steps already done).
subprocess.run(["bash", "scripts/setup_shopbench.sh", str(SHOPBENCH_WORK)], check=True,
               env={**os.environ, "PYTHON": sys.executable})
harness = Harness(SHOPBENCH_WORK.resolve())
if not harness.search_ok():
    harness.start_search(workers=8)

# The training problems from 01a, one file per task for the harness.
GEN_DIR = (SHOPBENCH_WORK / "out" / "bedrock-teacher").resolve()
GEN_DIR.mkdir(parents=True, exist_ok=True)
problems = {}
for t, rows in train_problems.items():
    problems[t] = GEN_DIR / f"problems_train_{t}.jsonl"
    write_jsonl(rows, problems[t])

# Published trajectories: unpack them into GEN_DIR so the resume-safe generation below finds them done.
if USE_PUBLISHED_TEACHER_TRAJECTORIES:
    import gzip, shutil
    PUBLISHED = Path("teacher_trajectories/glm-5.3")
    assert BEDROCK_TEACHER == "us.zai.glm-5.3" and TEACHER_SAMPLES <= 2, \
        "the published trajectories are GLM-5.3, 2 samples per problem: set BEDROCK_TEACHER/TEACHER_SAMPLES to match"
    for k in range(TEACHER_SAMPLES):
        for t in problems:
            dst = GEN_DIR / f"rollout_teacher_s{k}_{t}.jsonl"
            if not dst.exists():
                with gzip.open(PUBLISHED / f"{dst.name}.gz", "rb") as fi, open(dst, "wb") as fo:
                    shutil.copyfileobj(fi, fo)
    print(f"Published GLM-5.3 trajectories unpacked into {GEN_DIR}")

url = relay.start_relay("teacher", BEDROCK_TEACHER, 8401, REGION, transport="bedrock", extra_fields=TEACHER_FIELDS)
jobs = {}
for k in range(TEACHER_SAMPLES):
    for t, p in problems.items():
        cfg, rollout = harness.write_rollout_cfg(GEN_DIR, f"teacher_s{k}", t, p, url, True, threads=4,
                                                 temperature=1.0, max_tokens=8192)
        c = json.loads(Path(cfg).read_text())
        c["model_config"]["top_p"] = 0.95      # GLM-5.3's evaluation setting (temperature 1.0, top_p 0.95)
        Path(cfg).write_text(json.dumps(c, indent=2))
        jobs[f"teacher_s{k}_{t}"] = (cfg, rollout, sum(1 for _ in open(p)))
harness.run_rollouts(jobs, poll_s=60)
relay.stop_relay("teacher")

# Rejection sampling: keep only trajectories the benchmark's own rules score as solved, then encode each step.
pairs = []
for name, (cfg, rollout, n) in jobs.items():
    t = name.rsplit("_", 1)[1]
    pq = GEN_DIR / f"perquery_{name}.json"
    for _ in range(3):   # the scorer can abort at exit after writing its result; retry if it didn't
        res = subprocess.run([str(harness.py), str(Path("scripts/score_rollouts.py").resolve()), "--task", t,
                              "--rollouts", str(rollout), "--problems", str(problems[t]), "--per-query", str(pq)],
                             cwd=harness.sb, env=harness.env, capture_output=True, text=True)
        if pq.exists():
            break
    else:
        print(f"{name}: score_rollouts.py wrote no result (exit {res.returncode}); stderr tail:\n{res.stderr[-2000:]}")
    solved = {q for q, v in json.loads(pq.read_text()).items() if v["solved"]}
    pairs += sft_rows(rollout, solved, INCLUDE_REASONING)
    print(f"{name}: {len(solved)} / {n} solved")

# Split by query, so no query is in both train and validation.
queries = sorted({q for q, _ in pairs})
random.seed(42); random.shuffle(queries)
val_q = set(queries[:max(1, len(queries) // 20)])
train = [r for q, r in pairs if q not in val_q]
val = [r for q, r in pairs if q in val_q]
random.shuffle(train)
print(f"Bedrock teacher: train {len(train)} / val {len(val)} examples from {len(queries)} solved queries "
      f"({'with' if INCLUDE_REASONING else 'without'} reasoning)")

### Filter by length, then write the dataset files

The trainer enforces a maximum sequence length (`dataset_max_len`, set in step 02) and **silently left-truncates** anything longer — which would chop the instructions off the front of an example and corrupt the target. To avoid that, we measure each example's token count and **drop** any that exceed the limit, rather than letting them be truncated. The cell also prints the length distribution (median / p99 / max) so you can sanity-check the limit, shows one fully-rendered example as the trainer will see it, and writes the train/val splits to local `.jsonl` files for upload.

In [ ]:
# Drop examples longer than dataset_max_len: the trainer truncates from the left
MAX_TOKENS = 16384  # = dataset_max_len in step 02

def token_lengths(rows, chunk=1000):
    # Render once, then tokenize in batched chunks (fast-tokenizer parallelism) to bound peak memory.
    lens = []
    for i in range(0, len(rows), chunk):
        texts = [tok.apply_chat_template(r["messages"], tokenize=False) for r in rows[i:i + chunk]]
        lens.extend(len(ids) for ids in tok(texts)["input_ids"])
    return lens

for name, rows in (("train", train), ("val", val)):
    lens = token_lengths(rows)
    keep = [r for r, n in zip(rows, lens) if n <= MAX_TOKENS]
    s = sorted(lens)
    print(f"{name}: tokens per example median {s[len(s) // 2]}, p99 {s[int(len(s) * 0.99)]}, max {s[-1]}; "
          f"\n dropped {len(rows) - len(keep)} longer than {MAX_TOKENS}")
    rows[:] = keep

print(f"--- example (as the trainer renders it) ---\n {tok.apply_chat_template(train[0]['messages'], tokenize=False)}")

# Write the files to local disk under data/ (git-ignored).
from pathlib import Path
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)
TRAIN_FILE, VAL_FILE = str(DATA_DIR / "seqkd_train.jsonl"), str(DATA_DIR / "seqkd_val.jsonl")

write_jsonl(train, TRAIN_FILE)
write_jsonl(val, VAL_FILE)

### Upload and register the datasets

Upload the two train and validation files to S3, then register each with `DataSet.create`. Registration produces a **versioned dataset ARN** in the SageMaker AI Registry rather than a bare S3 path. The serverless `SFTTrainer` consumes these ARNs directly, which gives you lineage and reproducibility — every training job records exactly which dataset version it trained on.

In [ ]:
from sagemaker.ai_registry.dataset import DataSet
from pathlib import Path

s3 = boto3.client("s3", region_name=REGION)
# TRAIN_FILE/VAL_FILE are local paths under data/; use just the basename for the S3 key so the
# object lands at {PREFIX}/data/<name>.jsonl (not a doubled data/data/).
TRAIN_KEY = f"{PREFIX}/data/{Path(TRAIN_FILE).name}"
VAL_KEY   = f"{PREFIX}/data/{Path(VAL_FILE).name}"
s3.upload_file(TRAIN_FILE, BUCKET, TRAIN_KEY)
s3.upload_file(VAL_FILE,   BUCKET, VAL_KEY)
TRAIN_S3 = f"s3://{BUCKET}/{TRAIN_KEY}"
VAL_S3   = f"s3://{BUCKET}/{VAL_KEY}"

train_dataset = DataSet.create(name="seqkd-shopping-train", source=TRAIN_S3)
val_dataset   = DataSet.create(name="seqkd-shopping-val",   source=VAL_S3)
print("train:", train_dataset.arn)
print("val:  ", val_dataset.arn)

## 02. Train the student with the serverless `SFTTrainer`

As noted earlier, SeqKD reduces to ordinary supervised fine-tuning on the teacher's trajectories — so training is a standard `SFTTrainer` job. The serverless trainer provisions and tears down the compute for you; you only define the model, data, and hyperparameters.

The cell below creates the trainer and a **Model Package Group** — a named container in the Model Registry that will hold the resulting model version. We train with **LoRA** (low-rank adapters), which is far cheaper than full fine-tuning and sufficient for this task.

> **Licensing:** `accept_eula=True` accepts the base model's license. Review the model's terms before running. Call `SFTTrainer.list_supported_models()` to see which base models serverless training supports.

In [ ]:
from sagemaker.train.sft_trainer import SFTTrainer
from sagemaker.train.common import TrainingType
from sagemaker.core.resources import ModelPackageGroup

MPG_NAME = "seqkd-qwen35-4b-shopping"
try:
    model_package_group = ModelPackageGroup.create(model_package_group_name=MPG_NAME)
except Exception as e:
    print("reusing existing group due to:", type(e).__name__)
    model_package_group = MPG_NAME          # an existing name or ARN is also accepted

# SFTTrainer.list_supported_models() lists the models serverless training supports

sft_trainer = SFTTrainer(
    model="huggingface-vlm-qwen3-5-4b",     # Qwen3.5-4B
    training_type=TrainingType.LORA,        # Use TrainingType.FULL for full fine-tuning
    model_package_group=model_package_group,
    mlflow_experiment_name="seqkd-shopping-exp",
    training_dataset=train_dataset.arn,
    validation_dataset=val_dataset.arn,
    s3_output_path=f"{S3_ROOT}/output/",
    accept_eula=True,                       #To accept licensing for the models
)

### Hyperparameters

Each supported model + training type ships with a tuned **recipe default**. The first cell prints those defaults; the second overrides only what we want to change for this run. Set overrides **after** creating the trainer — recreating the trainer resets them.

The overrides used here and why:

- `dataset_max_len = 16384` — matches the length filter from step 01 so long multi-step episodes aren't truncated.
- `max_epochs = 3` — the teacher's success-filtered set is small (~2.6k examples).
- `global_batch_size = 64`, `learning_rate = 1e-4`, `lr_warmup_steps_ratio = 0.03` — stable settings for LoRA SFT at this scale.
- `lora_rank = 64`, `lora_alpha = 256` (α/rank = 4) — higher-capacity adapters, which this reference configuration found improves tool-use success.
- `train_val_split_ratio = 0.9` — how the trainer splits within the training channel for its internal eval.

Any unsupported value raises an error that reports the allowed range, so it's safe to experiment.

In [ ]:
from pprint import pprint
print("Default hyperparameters")
pprint(sft_trainer.hyperparameters.to_dict())

In [ ]:
sft_trainer.hyperparameters.dataset_max_len       = 16384
sft_trainer.hyperparameters.max_epochs            = 3
sft_trainer.hyperparameters.global_batch_size     = 64
sft_trainer.hyperparameters.learning_rate         = 0.0001
sft_trainer.hyperparameters.lora_rank             = 64
sft_trainer.hyperparameters.lora_alpha            = 256     # alpha/rank = 4
sft_trainer.hyperparameters.train_val_split_ratio = 1.0
sft_trainer.hyperparameters.lr_warmup_steps_ratio = 0.03
pprint(sft_trainer.hyperparameters.to_dict())

### Validate, then launch training

Always **dry-run first**: `train(dry_run=True)` validates your IAM permissions, dataset ARNs, and hyperparameters without provisioning any compute — a fast way to catch misconfiguration before paying for a job.

Once the dry run passes, the next cell launches the real job. `wait=True` blocks and streams logs until completion; use `wait=False` if you'd rather submit and poll later. Training is serverless, so there's no instance to manage. Metrics stream to the MLflow experiment set on the trainer (`mlflow_experiment_name`).

In [ ]:
sft_trainer.train(dry_run=True)
print("Dry run passed - configuration is valid.")

In [ ]:
training_job = sft_trainer.train(wait=True)
print(f"Job Name: {training_job.training_job_name} Job Status: {training_job.training_job_status}")

### Inspect the training loss curve

Once the job completes, plot the loss from MLflow to sanity-check the run. The MLflow app, experiment, and run are resolved **at run time** and recorded on the training job, so the cell below reads them off the job rather than hardcoding anything (for the reference run that resolves to experiment `seqkd-shopping-exp`, run `huggingface-vlm-qwen3-5-4b-sft-...`).

A healthy run shows **training loss trending down and flattening** as the model learns the teacher's behavior.

> **On validation loss:** this recipe logs the *training* loss (`train/loss`) but does not emit a separate validation-loss series to MLflow, so there's no validation curve to overlay here. The authoritative generalization signal for this workflow is the **held-out ASR + IFEval** evaluation in 3-evaluate-shopping-agent notebook — those run on held-out queries the model never trained on.

In [ ]:
import matplotlib.pyplot as plt
from sagemaker.core.resources import TrainingJob

def plot_training_loss(trainer):
    job_name = getattr(trainer, "_latest_training_job", None) or training_job.training_job_name
    job_name = job_name if isinstance(job_name, str) else job_name.training_job_name
    job = TrainingJob.get(training_job_name=job_name)

    cfg, det = getattr(job, "mlflow_config", None), getattr(job, "mlflow_details", None)
    arn    = getattr(cfg, "mlflow_resource_arn", None)
    run_id = getattr(det, "mlflow_run_id", None)
    if not arn or not run_id:
        print("MLflow run not available yet (job still running or MLflow not configured).")
        return

    import mlflow
    from mlflow.tracking import MlflowClient
    mlflow.set_tracking_uri(arn)
    client = MlflowClient()
    print(f"experiment: {getattr(cfg, 'mlflow_experiment_name', '?')}  run: {getattr(cfg, 'mlflow_run_name', '?')}")

    loss = sorted(client.get_metric_history(run_id, "train/loss"), key=lambda h: h.step)
    if not loss:
        print(f"No 'train/loss' logged on run {run_id} yet.")
        return

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot([h.step for h in loss], [h.value for h in loss], linewidth=2, label="train/loss")
    ax.set_xlabel("Global step"); ax.set_ylabel("Loss")
    ax.set_title(f"SeqKD training loss — {job_name}", fontweight="bold")
    ax.grid(True, alpha=0.3); ax.legend()
    plt.tight_layout(); plt.show()

try:
    plot_training_loss(sft_trainer)
except Exception as e:
    print("Could not fetch loss curve:", type(e).__name__, e)

## 03. Retrieve the trained Model Package

When the job finishes it registers a **Model Package** in the group created in step 02. For a LoRA run, SageMaker has already **merged the adapter into the base weights**, so the package is a complete, ready-to-deploy model — no separate merge step needed.

This cell fetches the Model Package ARN from the finished training job and prints its status.

In [ ]:
from sagemaker.core.resources import TrainingJob, ModelPackage

job = TrainingJob.get(training_job_name=training_job.training_job_name)
MODEL_PKG_ARN = job.output_model_package_arn
# The adapter in the job's output, used by the evaluation notebook if no Model Package was registered
# (registration fails silently when the account's model-package quota is full).
ADAPTER_S3_URI = job.model_artifacts.s3_model_artifacts.rstrip("/") + "/checkpoints/hf/"
if isinstance(MODEL_PKG_ARN, str) and MODEL_PKG_ARN:
    print(f"Output Model Package ARN: {MODEL_PKG_ARN}")
    print(f"status: {ModelPackage.get(model_package_name=MODEL_PKG_ARN).model_package_status}")
else:
    MODEL_PKG_ARN = ""
    print("No Model Package was registered (check the account's model-package quota); the evaluation notebook "
          f"will use the adapter from the job output: {ADAPTER_S3_URI}")

# To find the package later without the job object:
#   aws sagemaker list-model-packages --model-package-group-name seqkd-qwen35-4b-shopping \
#     --sort-by CreationTime --sort-order Descending \
#     --query 'ModelPackageSummaryList[0].ModelPackageArn'

In [ ]:
%store MODEL_PKG_ARN
%store ADAPTER_S3_URI
%store INCLUDE_REASONING

## 04. Evaluate the student

Evaluate the fine-tuned student in two companion notebooks, in order — each takes the Model Package ARN from step 03:

1. **`2-tool-call-accuracy-gate.ipynb`** (run first) — a cheap, fast **gate**. A managed SageMaker **Custom Scorer** job measures per-step tool-call accuracy: given the teacher's prompt, does the student emit the right tool call? If the student doesn't beat the base here, fix training before running the complete evaluation. This saves the time and cost of a complete evaluation run.
2. **`3-evaluate-shopping-agent.ipynb`** — the full evaluation, which runs two complementary measures:
   - **Agent Success Rate (ASR)** — the headline metric. It runs the student through the full ShoppingBench agent loop on **held-out** problems and scores whether its final recommendation satisfies every constraint. This measures the capability we distilled.
   - **IFEval (regression check)** — general instruction-following, run to confirm that specializing the model for shopping tool-use didn't degrade its instruction-following ability.

Comparing the fine-tuned student against the base model on these axes tells you both how much task skill you gained and whether you preserved general competence.